In [ ]:
from timesfm import TimesFmCheckpoint, TimesFmHparams
from timesfm import timesfm_torch as ttfm
from timesfm import pytorch_patched_decoder as ppd
import torch


def get_model(
    batch_size: int,
    horizon_len: int,
    context_len: int,
    repo_id: str = "google/timesfm-2.0-500m-pytorch",
) -> ppd.PatchedTimeSeriesDecoder:
    torch.cuda.empty_cache()

    hparams = TimesFmHparams(
        backend="gpu" if torch.cuda.is_available() else "cpu",
        per_core_batch_size=batch_size,
        horizon_len=horizon_len,
        context_len=context_len,
        num_layers=50,  # fixed for 500m model
        use_positional_embedding=True,  # RoPE
        point_forecast_mode="mean",  # head output
    )

    tfm = ttfm.TimesFmTorch(
        hparams=hparams, checkpoint=TimesFmCheckpoint(huggingface_repo_id=repo_id)
    )

    model: ppd.PatchedTimeSeriesDecoder | None = tfm._model

    if model is None:
        raise ValueError("Model is None")

    return model

In [ ]:
from fusiontimeseries.lib.config import FTSConfig

fts_config = FTSConfig(
    context_length=512,
    prediction_length=128,
    pred_tail_timestamps=80,
    batch_size=128,
    padding_value=0.0,  # chronos2 has NaN as padding value
    padding_mask_default=0.0,
    padding_mask_indicator=1.0,
)

In [ ]:
# Zeroshot Benchmarker

from datetime import datetime
import json
from pathlib import Path
from typing import Literal
from torch import Tensor
from fusiontimeseries.lib.benchmarking import rmse_with_standard_error
from fusiontimeseries.lib.config import FTSConfig
from fusiontimeseries.lib.dataset import FluxData, TimeseriesDataset
import numpy as np

type PerformanceData = dict[str, dict[int, list[float]]]
type BenchmarkData = dict[Literal["ood", "id"], dict[int, FluxData]]


class ZeroshotBenchmarker:
    BENCHMARK_FLUX_TS_LENGTH: int = 267
    CONTEXT_LENGTH_STEP: int = 20

    def __init__(self, fts_config: FTSConfig) -> None:
        self.fts_config = fts_config

        self.benchmark_data: BenchmarkData = (
            TimeseriesDataset.get_benchmark_flux_traces(config=self.fts_config)
        )
        self.train_data: list[FluxData] = TimeseriesDataset.load_flux_data(
            config=self.fts_config
        )
        self.model = None
        self.model_slug: str | None = None

        # set seed for reproducibility
        torch.manual_seed(fts_config.random_seed)
        torch.cuda.manual_seed_all(fts_config.random_seed)
        np.random.seed(fts_config.random_seed)

    @torch.inference_mode()
    def call_model(self, ctx: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError(
            "call_model method not implemented. Please implement this method to call the model for predictions."
        )

    def autoregressive_rollout(
        self, ctx: torch.Tensor, rollout_horizon: int
    ) -> np.ndarray:
        while len(ctx) < rollout_horizon:
            forecast = self.call_model(ctx)
            if len(forecast) > self.fts_config.prediction_length:
                print(
                    f"Warning: Model returned more predictions ({len(forecast)}) than expected ({self.fts_config.prediction_length}). Truncating to expected length."
                )
            ctx = torch.cat((ctx, forecast), dim=0)
        return ctx[:rollout_horizon].cpu().numpy()

    def sample_rollout(
        self, start_context_length: int, flux_data: FluxData
    ) -> np.ndarray:
        time_series: np.ndarray = np.array(flux_data.energy_flux)
        ctx = torch.tensor(time_series[:start_context_length], dtype=torch.float32)
        forecast: np.ndarray = self.autoregressive_rollout(
            ctx, rollout_horizon=len(time_series)
        )
        return forecast

    def run(
        self, data: dict[str, dict[int, FluxData]] | None = None
    ) -> PerformanceData:
        if self.model is None:
            raise ValueError(
                "Model not set. Please set the model before running the benchmark."
            )

        max_context_length: int = (
            self.BENCHMARK_FLUX_TS_LENGTH - self.fts_config.pred_tail_timestamps
        )
        start_context_length: int = max_context_length % self.CONTEXT_LENGTH_STEP

        _data = data or self.benchmark_data
        performance: PerformanceData = {**{namespace: {} for namespace in _data.keys()}}
        for namespace, samples in _data.items():
            print(f"Evaluating Namespace: {namespace} with {len(samples)} samples")
            for context_length in range(
                start_context_length, max_context_length + 1, self.CONTEXT_LENGTH_STEP
            ):
                true_tail_mean: list[float] = []
                pred_tail_mean: list[float] = []
                for _, flux_data in samples.items():
                    forecast: np.ndarray = self.sample_rollout(
                        start_context_length=context_length,
                        flux_data=flux_data,
                    )

                    pred_tail_mean.append(
                        forecast[-self.fts_config.pred_tail_timestamps :].mean()
                    )
                    true_tail_mean.append(
                        float(
                            np.mean(
                                flux_data.energy_flux[
                                    -self.fts_config.pred_tail_timestamps :
                                ],
                                dtype=np.float32,
                            )
                        )
                    )
                rsme, rsme_se = rmse_with_standard_error(
                    np.array(true_tail_mean), np.array(pred_tail_mean)
                )
                performance[namespace][context_length] = [rsme, rsme_se]
        return performance

    def save_results(self, performance: PerformanceData) -> None:
        if self.model_slug is None:
            raise ValueError(
                "Model slug not set. Please set the model slug before saving results."
            )

        model_name_clean = self.model_slug.replace("/", "_")
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        # Save results to JSON
        data_dir = Path(".").resolve() / "results" / model_name_clean
        data_dir.mkdir(parents=True, exist_ok=True)
        results_file = (
            data_dir / f"{timestamp}_{model_name_clean}_zeroshot_performance.json"
        )
        with open(results_file, "w") as f:
            json.dump(performance, f, indent=2)

        print(f"Results saved to: {results_file}")

In [ ]:
from typing import Any, Callable


class TimesFM2p0500MBenchmarker(ZeroshotBenchmarker):
    def __init__(
        self,
        fts_config: FTSConfig,
        forward_transform: Callable[
            [torch.Tensor, torch.Tensor],
            tuple[torch.Tensor, tuple[torch.Tensor, torch.Tensor]],
        ]
        | None = None,
        reverse_transform: Callable[
            [torch.Tensor, tuple[torch.Tensor, torch.Tensor]], torch.Tensor
        ]
        | None = None,
    ) -> None:
        super().__init__(fts_config)
        self.model_slug = "google/timesfm-2.0-500m-pytorch"
        if forward_transform is not None and reverse_transform is not None:
            ppd.PatchedTimeSeriesDecoder._forward_transform = forward_transform  # type: ignore
            ppd.PatchedTimeSeriesDecoder._reverse_transform = reverse_transform  # type: ignore

        self.model = get_model(
            batch_size=fts_config.batch_size,
            horizon_len=fts_config.prediction_length,
            context_len=fts_config.context_length,
        )
        self.device = self.model.freq_emb.weight.device
        self.model.eval()

    @torch.inference_mode()
    def call_model(self, ctx: Tensor) -> Tensor:
        # ctx shape: (L,)
        L = len(ctx)
        context_start_idx = self.fts_config.context_length - L

        ############### Context #################
        context = torch.full(
            size=(self.fts_config.context_length,),
            fill_value=self.fts_config.padding_value,  # 0.0
            dtype=torch.float32,
        )
        context[context_start_idx:] = ctx

        ################ Padding Mask #################
        context_mask = torch.full_like(
            context,
            fill_value=self.fts_config.padding_mask_default,  # 0.0
            dtype=torch.float32,
        )
        # assign padding indicator to PADDED positions
        context_mask[:context_start_idx] = self.fts_config.padding_mask_indicator  # 1.0
        freq = torch.tensor([0], dtype=torch.long)

        # prediction shape: (batch_size, num_patches, prediction_length, mean + quantiles)
        predictions: torch.Tensor = self.model(
            input_ts=context.unsqueeze(0).to(self.device),  # (1, context_length)
            input_padding=context_mask.unsqueeze(0).to(
                self.device
            ),  # (1, context_length)
            freq=freq.unsqueeze(0).to(self.device),  # (1, 1)
        )
        point_forecast = predictions[0, -1, :, 0]  # (prediction_length,)
        return point_forecast.detach().cpu()

In [ ]:
benchmark_data: dict[str, Any] = TimeseriesDataset.get_benchmark_flux_traces(
    config=fts_config
)  # type: ignore
flux_data: list[FluxData] = TimeseriesDataset.load_flux_data(config=fts_config)
validation_data = [flux_data for flux_data in flux_data if flux_data.is_validation]
validation_data = TimeseriesDataset.subsample_flux_data(
    validation_data,
    window=fts_config.subsample_factor,
    stride=fts_config.subsample_factor,
)
print(
    f"Loaded {len(flux_data)} flux data samples, with {len(validation_data)} validation samples."
)
benchmark_data["val"] = {flux_data.idx: flux_data for flux_data in validation_data}

# train data
# train_data = [flux_data for flux_data in flux_data if flux_data.is_train]
# train_data = TimeseriesDataset.subsample_flux_data(train_data, window=fts_config.subsample_factor, stride=fts_config.subsample_factor)
# benchmark_data["train"] = {flux_data.idx: flux_data for flux_data in train_data}

In [ ]:
for namespace, samples in benchmark_data.items():
    for idx, flux_data in samples.items():
        print(
            f"Namespace: {namespace}, Sample ID: {idx}, Energy Flux Length: {len(flux_data.energy_flux)}"
        )

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def _no_norm_forward_transform(
    self, inputs: torch.Tensor, patched_pads: torch.Tensor
) -> tuple[torch.Tensor, tuple[torch.Tensor, torch.Tensor]]:
    """Input is of shape [B, N, P]."""
    mu = torch.tensor([0.0], device=inputs.device)
    sigma = torch.tensor([1.0], device=inputs.device)
    outputs = inputs  # no normalization
    return outputs, (mu, sigma)


def _no_norm_reverse_transform(
    self, outputs: torch.Tensor, stats: tuple[torch.Tensor, torch.Tensor]
) -> torch.Tensor:
    return outputs  # no normalization, so reverse is identity


def _global_forward_transform(
    self, inputs: torch.Tensor, patched_pads: torch.Tensor
) -> tuple[torch.Tensor, tuple[torch.Tensor, torch.Tensor]]:
    """Input is of shape [B, N, P]."""
    masked_inputs = inputs.masked_fill(patched_pads.bool(), float("nan"))
    mu = torch.nanmean(masked_inputs).unsqueeze(0)
    sigma = torch.sqrt(torch.nanmean((masked_inputs - mu) ** 2)).unsqueeze(0)
    sigma = torch.where(
        sigma < self.config.tolerance,
        torch.tensor(1.0, dtype=sigma.dtype, device=sigma.device),
        sigma,
    )

    # Normalize each patch
    outputs = (inputs - mu[:, None, None]) / sigma[:, None, None]

    # clamp norm output to max value to prevent instability
    outputs = torch.where(
        torch.abs(inputs - self.config.pad_val) < self.config.tolerance,
        torch.tensor(self.config.pad_val, dtype=outputs.dtype, device=outputs.device),
        outputs,
    )
    return outputs, (mu, sigma)


def _global_reverse_transform(
    self, outputs: torch.Tensor, stats: tuple[torch.Tensor, torch.Tensor]
) -> torch.Tensor:
    mu, sigma = stats
    return outputs * sigma[:, None, None, None] + mu[:, None, None, None]


def _instancenorm_forward_transform(
    self, inputs: torch.Tensor, patched_pads: torch.Tensor
) -> tuple[torch.Tensor, tuple[torch.Tensor, torch.Tensor]]:
    """Input is of shape [B, N, P]."""
    masked_inputs = inputs.masked_fill(patched_pads.bool(), float("nan"))
    mu = torch.nanmean(masked_inputs).unsqueeze(0)
    sigma = torch.sqrt(torch.nanmean((masked_inputs - mu) ** 2)).unsqueeze(0)
    sigma = torch.where(
        sigma < self.config.tolerance,
        torch.tensor(1.0, dtype=sigma.dtype, device=sigma.device),
        sigma,
    )
    # print("mu", mu.item(), "sigma", sigma.item())
    # if inputs.max() > 1000.0:
    #     print("Warning: Input values are very large, which may cause instability in normalization.")
    #     print("Input stats - min:", inputs.min().item(), "max:", inputs.max().item(), "mean:", inputs.mean().item(), "std:", inputs.std().item())

    # Normalize
    scaled_inputs = (inputs - mu[:, None, None]) / sigma[:, None, None]

    # Apply arcsinh transformation
    # print("scaled inputs", scaled_inputs.min(), scaled_inputs.max())

    outputs = torch.arcsinh(
        scaled_inputs.masked_fill(patched_pads.bool(), 0.0)
    )  # arcsinh(0) = 0, so this won't affect the padded values

    # print("arcsinh outputs", outputs.min(), outputs.max())

    outputs = torch.where(
        torch.abs(inputs - self.config.pad_val) < self.config.tolerance,
        torch.tensor(self.config.pad_val, dtype=outputs.dtype, device=outputs.device),
        outputs,
    )
    print("final inputs", outputs.min(), outputs.max())

    # print(outputs.shape, mu.shape, scale.shape)

    return outputs, (mu, sigma)


def _instancenorm_reverse_transform(
    self, outputs: torch.Tensor, stats: tuple[torch.Tensor, torch.Tensor]
) -> torch.Tensor:
    mu, sigma = stats
    # print("mu", mu.item(), "sigma", sigma.item())
    # Inverse arcsinh transformation
    print("outputs before inverse transform", outputs.min(), outputs.max())
    scaled_inputs = torch.sinh(outputs)
    print(
        "scaled outputs after inverse transform",
        scaled_inputs.min(),
        scaled_inputs.max(),
    )
    # Denormalize
    inputs_f32 = scaled_inputs * sigma[:, None, None] + mu[:, None, None]
    return inputs_f32.to(outputs.dtype)

In [ ]:
timesfm_performance_base = TimesFM2p0500MBenchmarker(fts_config=fts_config).run(
    benchmark_data
)
timesfm_performance_base

In [ ]:
timesfm_performance_nonorm = TimesFM2p0500MBenchmarker(
    fts_config=fts_config,
    forward_transform=_no_norm_forward_transform,
    reverse_transform=_no_norm_reverse_transform,
).run(benchmark_data)
timesfm_performance_nonorm

In [ ]:
timesfm_performance_globalmeanstd = TimesFM2p0500MBenchmarker(
    fts_config=fts_config,
    forward_transform=_global_forward_transform,
    reverse_transform=_global_reverse_transform,
).run(benchmark_data)
timesfm_performance_globalmeanstd

In [ ]:
timesfm_performance_instancenorm = TimesFM2p0500MBenchmarker(
    fts_config=fts_config,
    forward_transform=_instancenorm_forward_transform,
    reverse_transform=_instancenorm_reverse_transform,
).run(benchmark_data)
timesfm_performance_instancenorm

In [ ]:
# plot results on x is context length and y is rmse with error bars for each normalization method
# make three plots for each namespace (ood, id, val) with context length on x-axis and rmse on y-axis, with error bars for standard error, and different lines for each normalization method (base, none, global_mean_std)
from matplotlib import pyplot as plt

namespaces = benchmark_data.keys()
fig, axes = plt.subplots(
    1, len(namespaces), figsize=(len(namespaces) * 4, 5), sharey=True
)

# Define jitter offsets for each normalization method
jitter_offset = 4
jitter = {
    "base": -jitter_offset,
    "nonorm": 0,
    "globalmeanstd": jitter_offset,
    "arcsinhnorm": jitter_offset * 2,
}

for i, namespace in enumerate(namespaces):
    ax = axes[i]
    base_performance = timesfm_performance_base[namespace]
    nonorm_performance = timesfm_performance_nonorm[namespace]
    globalmeanstd_performance = timesfm_performance_globalmeanstd[namespace]
    arcsinhnorm_performance = timesfm_performance_instancenorm[namespace]

    context_lengths = sorted(base_performance.keys())
    base_rmse = [base_performance[cl][0] for cl in context_lengths]
    base_se = [base_performance[cl][1] for cl in context_lengths]

    nonorm_rmse = [nonorm_performance[cl][0] for cl in context_lengths]
    nonorm_se = [nonorm_performance[cl][1] for cl in context_lengths]

    globalmeanstd_rmse = [globalmeanstd_performance[cl][0] for cl in context_lengths]
    globalmeanstd_se = [globalmeanstd_performance[cl][1] for cl in context_lengths]

    arcsinhnorm_rmse = [arcsinhnorm_performance[cl][0] for cl in context_lengths]
    arcsinhnorm_se = [arcsinhnorm_performance[cl][1] for cl in context_lengths]

    # Apply jitter to context lengths
    base_ctx = [cl + jitter["base"] for cl in context_lengths]
    nonorm_ctx = [cl + jitter["nonorm"] for cl in context_lengths]
    globalmeanstd_ctx = [cl + jitter["globalmeanstd"] for cl in context_lengths]
    arcsinhnorm_ctx = [cl + jitter["arcsinhnorm"] for cl in context_lengths]

    ax.errorbar(base_ctx, base_rmse, yerr=base_se, label="Base Norm", marker="o")
    ax.errorbar(nonorm_ctx, nonorm_rmse, yerr=nonorm_se, label="No Norm", marker="o")
    ax.errorbar(
        globalmeanstd_ctx,
        globalmeanstd_rmse,
        yerr=globalmeanstd_se,
        label="Global Mean/Std",
        marker="o",
    )
    ax.errorbar(
        arcsinhnorm_ctx,
        arcsinhnorm_rmse,
        yerr=arcsinhnorm_se,
        label="ArcSinhNorm",
        marker="o",
    )

    ax.set_title(f"Namespace: {namespace}")
    ax.set_xlabel("Context Length")
    ax.set_xticks(context_lengths)
    ax.set_ylim(0, 200)
    if i == 0:
        ax.set_ylabel("RMSE")
    ax.legend()